# UCS420: Cognitive Computing — Assignment 4

## Q1: Build Your Personalized Knowledge Base

In [1]:
import pandas as pd

roll_number = "1024170213"

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]
last_two_digits = [int(d) for d in roll_number[-2:]]
category_map = ["billing", "account", "general"]
personalized_entries = []
for d in last_two_digits:
    cat = category_map[d % 3]
    print(f"digit {d} -> category[{d} % 3] = {cat}")
    personalized_entries.append(cat)
print("Categories for personalized entries:", personalized_entries)
account_entry = {
    "question": "how do i update my registered mobile number",
    "answer": "Go to Profile > Contact Details > Update Mobile Number, then verify with OTP.",
    "keywords": "mobile number update",
    "category": "account"
}
billing_entry = {
    "question": "how do i get a refund for an overpayment",
    "answer": "Refunds are processed within 5-7 business days to your original payment method.",
    "keywords": "refund overpayment billing",
    "category": "billing"
}
all_entries = fixed_entries + [account_entry, billing_entry]
df = pd.DataFrame(all_entries)
df

digit 1 -> category[1 % 3] = account
digit 3 -> category[3 % 3] = billing
Categories for personalized entries: ['account', 'billing']


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my registered mobile number,Go to Profile > Contact Details > Update Mobil...,mobile number update,account
5,how do i get a refund for an overpayment,Refunds are processed within 5-7 business days...,refund overpayment billing,billing


## Q2: Generate and Score a Hypothesis

In [2]:
def score_entries(query, df):
    query_words = set(query.lower().split())
    scores = []
    for _, row in df.iterrows():
        entry_keywords = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())
        keyword_match = len(query_words & entry_keywords)
        question_match = len(query_words & question_words)
        confidence = keyword_match + 0.5 * question_match
        scores.append(confidence)

    df_scored = df.copy()
    df_scored["confidence"] = scores
    result = df_scored[df_scored["confidence"] > 0].sort_values("confidence", ascending=False)
    return result.reset_index(drop=True)
score_entries("fee payment", df)


,question,answer,keywords,category,confidence
0,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,2.5
1,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing,1.5


## Q3: `same_category(category_name, df)`

In [3]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"]
result_q3 = same_category("account", df)
print("Questions in category 'account':")
print(result_q3.to_string(index=False))


Questions in category 'account':
                      how to reset password
how do i update my registered mobile number


## Q4: Add a New Keyword and Save to CSV

In [4]:
target_question = "how to reset password"
new_keyword = input(f"Enter a new keyword to add to the entry \"{target_question}\": ").strip()
idx = df[df["question"] == target_question].index[0]
df.loc[idx, "keywords"] = df.loc[idx, "keywords"] + " " + new_keyword
csv_filename = f"{roll_number}_faq_data.csv"
df.to_csv(csv_filename, index=False)
print(f"Updated keywords: {df.loc[idx, 'keywords']}")
print(f"Saved updated DataFrame to: {csv_filename}")
df

Enter a new keyword to add to the entry "how to reset password":  ff


Updated keywords: password reset login ff
Saved updated DataFrame to: 1024170213_faq_data.csv


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login ff,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my registered mobile number,Go to Profile > Contact Details > Update Mobil...,mobile number update,account
5,how do i get a refund for an overpayment,Refunds are processed within 5-7 business days...,refund overpayment billing,billing


## Q5: FAQ Entries per Category (groupby)

In [ ]:
category_counts = df.groupby("category").size()
print("Number of FAQ entries per category:")
print(category_counts)


## Q6: Handling Ties in Scoring

In [5]:
def score_entries_with_ties(query, df):
    query_words = set(query.lower().split())
    scores = []
    for _, row in df.iterrows():
        entry_keywords = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())
        keyword_match = len(query_words & entry_keywords)
        question_match = len(query_words & question_words)
        confidence = keyword_match + 0.5 * question_match
        scores.append(confidence)
    df_scored = df.copy()
    df_scored["confidence"] = scores
    matched = df_scored[df_scored["confidence"] > 0].sort_values("confidence", ascending=False)
    if matched.empty:
        print(f"Query: \"{query}\" -> No matching entries found.\n")
        return matched
    max_score = matched["confidence"].max()
    tied = matched[matched["confidence"] == max_score]

    print(f"Query: \"{query}\"")
    if len(tied) > 1:
        print(f"TIE DETECTED: {len(tied)} entries share the highest confidence score ({max_score}):")
        print(tied[["question", "answer", "category", "confidence"]].to_string(index=False))
    else:
        print(f"Top match (confidence {max_score}):")
        print(tied[["question", "answer", "category", "confidence"]].to_string(index=False))
    print()
    return matched
_ = score_entries_with_ties("fee", df)
_ = score_entries_with_ties("password reset", df)


Query: "fee"
TIE DETECTED: 2 entries share the highest confidence score (1.5):
              question                                     answer category  confidence
what is the annual fee                  The annual fee is Rs 500.  billing         1.5
 how can i pay the fee You can pay via UPI, card, or net banking.  billing         1.5

Query: "password reset"
Top match (confidence 3.0):
             question                           answer category  confidence
how to reset password Go to Settings > Reset Password.  account         3.0

